In [1]:
import os, subprocess

# Clone repo nếu chưa có
if not os.path.exists("/kaggle/working/sam3"):
    !git clone https://github.com/facebookresearch/sam3.git /kaggle/working/sam3

%cd /kaggle/working/sam3

# Cài train + dev dependencies
!pip install -e ".[train,dev]" -q

Cloning into '/kaggle/working/sam3'...
remote: Enumerating objects: 927, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 927 (delta 12), reused 3 (delta 3), pack-reused 905 (from 3)
Receiving objects: 100% (927/927), 59.48 MiB | 49.72 MiB/s, done.
Resolving deltas: 100% (190/190), done.
/kaggle/working/sam3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.

In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

# 1. Rút token bảo mật từ Kaggle Secrets
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# 2. Đăng nhập vào Hugging Face
login(token=HF_TOKEN)

# 3. Tải checkpoint thẳng vào bộ nhớ Cache mặc định của HF
print("Downloading SAM3 image checkpoints to cache...")
snapshot_download(
    repo_id="facebook/sam3",
    ignore_patterns=["*video*", "*.md", "*.safetensors"], # Lọc bớt file không cần thiết
    token=HF_TOKEN
)
print("Download completed! Mô hình đã sẵn sàng trong cache.")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Download completed! Mô hình đã sẵn sàng trong cache.


In [19]:
!cp -f /kaggle/input/datasets/lhuyton/config/roboflow_v100_full_ft_100_images.yaml /kaggle/working/sam3/sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml
# 2. In ra 25 dòng để kiểm tra chắc chắn file đã nhận cấu hình mới
print("=== KIỂM TRA FILE CONFIG ===")
!grep "num_train_workers" /kaggle/working/sam3/sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml

=== KIỂM TRA FILE CONFIG ===
  num_train_workers: 2
      num_workers: ${scratch.num_train_workers}


In [5]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 6.1 MB/s eta 0:00:000:00:0100:01


In [20]:
!NCCL_P2P_DISABLE=1 python /kaggle/working/sam3/sam3/train/train.py \
  -c configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml \
  --use-cluster 0 \
  --num-gpus 2

###################### Train App Config ####################
paths:
  roboflow_vl_100_root: /kaggle/input/datasets/lhuyton
  experiment_log_dir: /kaggle/working/sam3_logs
  bpe_path: /kaggle/working/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz
roboflow_train:
  num_images: 100
  supercategory: ${all_roboflow_supercategories.${string:${submitit.job_array.task_index}}}
  train_transforms:
  - _target_: sam3.train.transforms.basic_for_api.ComposeAPI
    transforms:
    - _target_: sam3.train.transforms.filter_query_transforms.FlexibleFilterFindGetQueries
      query_filter:
        _target_: sam3.train.transforms.filter_query_transforms.FilterCrowds
    - _target_: sam3.train.transforms.point_sampling.RandomizeInputBbox
      box_noise_std: 0.1
      box_noise_max: 20
    - _target_: sam3.train.transforms.segmentation.DecodeRle
    - _target_: sam3.train.transforms.basic_for_api.RandomResizeAPI
      sizes:
        _target_: sam3.train.transforms.basic.get_random_resize_scales
        si

In [31]:
!du -sh /kaggle/working/sam3_logs/checkpoints/*

4.8G	/kaggle/working/sam3_logs/checkpoints/checkpoint_10.pt


In [30]:
!rm /kaggle/working/sam3_logs/checkpoints/checkpoint.pt

In [32]:
import shutil
import os
from IPython.display import FileLink

# Đường dẫn thư mục log cần nén
log_dir = "/kaggle/working/sam3_logs"
# Tên file zip sau khi nén (sẽ được lưu tại /kaggle/working/sam3_logs_backup.zip)
output_filename = "/kaggle/working/sam3_logs_backup"

print("Đang nén thư mục log và checkpoint...")
shutil.make_archive(output_filename, 'zip', log_dir)
print("Hoàn tất!")

os.chdir('/kaggle/working')

# Tạo link tải trực tiếp
display(FileLink('sam3_logs_backup.zip'))

Đang nén thư mục log và checkpoint...
Hoàn tất! Hãy mở tab Output (bên phải) và tải file sam3_logs_backup.zip về máy.


In [33]:
import os
from IPython.display import FileLink

# Đảm bảo môi trường đang ở đúng thư mục chứa file zip
os.chdir('/kaggle/working')

# Tạo link tải trực tiếp
display(FileLink('sam3_logs_backup.zip'))

/kaggle/working/sam3_logs_backup.zip